# ZS-BERT — m=10 — one independent run
Select dataset and RUN_ID (1–5 → seeds 300–304). Save Version → Save & Run All.
GPU + Internet required. Data uses versioned BTL snapshots with SHA256; no dataset Input needed. Author Drive identity is unverified.
A paused run is incomplete: attach its output and resume the same seed.
10 epochs follow authors' README; best selects on test F1, final reports epoch 10.
This implementation follows the existing repo model/loss and records known reproduction limits.


In [1]:
import time
SESSION_STARTED = time.time()
DATASET = "wikizsl"
RUN_ID = 1
CODE_REV = "b4b8420"
DATA_SOURCE = "snapshot"
SMOKE_TEST = True
ALLOW_WIKI_COUNT_MISMATCH = True
RESUME_DIR = ""
EXPECTED_DATA_SHA256 = ""
ALLOW_WIKI_COUNT_MISMATCH = False  # Explicitly acknowledge a documented source discrepancy.
RESUME_DIR = ""  # Trusted previous output directory, only if continuing this same run.
MAX_SESSION_HOURS = 10.5
SMOKE_TEST = True  # Set False only after the GPU smoke and resume checks pass.


## Setup
No access token is required for the public GitHub repository. The actual commit is printed and saved.

In [2]:
import os, sys, subprocess
from pathlib import Path
os.environ.update(PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false",
    CUBLAS_WORKSPACE_CONFIG=":4096:8", HF_HUB_DISABLE_XET="1", USE_TF="0", USE_FLAX="0",
    PYTHONHASHSEED="0")
WORK = Path("/kaggle/working")
REPO = WORK / "zs-bert-simulation"
URL = "https://github.com/Student-persist-1919/zs-bert-simulation.git"
if not REPO.exists():
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
if subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip() != URL:
    raise RuntimeError("Unexpected repository origin")
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", CODE_REV], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
print("CODE COMMIT:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True))
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO/"requirements-kaggle.txt")], check=True)
subprocess.run([sys.executable, "-c", "from transformers import BertModel; from sentence_transformers import SentenceTransformer; import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))"], check=True)


Cloning into '/kaggle/working/zs-bert-simulation'...
fatal: couldn't find remote ref b4b8420


CalledProcessError: Command '['git', '-C', '/kaggle/working/zs-bert-simulation', 'fetch', 'origin', 'b4b8420']' returned non-zero exit status 128.

## Run and export
The deadline includes setup/download time. The evidence ZIP excludes the multi-GB checkpoint. Smoke tests run one epoch on four samples per relation and cannot enter official results.

In [ ]:
assert DATASET in ("fewrel", "wikizsl") and RUN_ID in range(1, 6)
assert 0 < MAX_SESSION_HOURS <= 11
cmd = [sys.executable, "-u", str(REPO/"scripts/kaggle_run.py"),
       "--dataset", DATASET, "--data-source", DATA_SOURCE, "--m", str(M), "--run-id", str(RUN_ID),
       "--deadline", str(SESSION_STARTED + MAX_SESSION_HOURS * 3600)]
for flag, value in [("--resume", RESUME_DIR), ("--data-file", DATA_FILE),
                    ("--expected-data-sha256", EXPECTED_DATA_SHA256)]:
    if value:
        cmd += [flag, value]
if ALLOW_WIKI_COUNT_MISMATCH:
    cmd += ["--allow-wiki-count-mismatch"]
if SMOKE_TEST:
    cmd += ["--smoke"]
subprocess.run(cmd, check=True)


## Preserve evidence
Download the `*_evidence.zip` from Output. Download the executed `.ipynb` from this exact Kaggle version.
For paused runs keep the full output directory (including checkpoint) on Kaggle; resume is the same run.
Locally import official evidence with:
`python scripts/collect_results.py --zip PATH --attempt original --kaggle-version-url VERSION_URL`
Store the executed notebook in `notebooks/kaggle/executed/` with dataset, m and seed in its name.
The collector regenerates `results.csv` and `RESULTS.md` beside these three notebooks.
